In [1]:
import os, itertools, importlib.util, time
import numpy as np, pandas as pd, torch
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

COMP = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
WORK = "/kaggle/working"
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
          "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
          "Contusion", "Fracture"]

t0 = time.time()
FILES, VOL_DIRS, LABEL_PATHS = {}, {}, []
for root, dirs, files in os.walk("/kaggle/input"):
    dirs[:] = [d for d in dirs if d != "competitions"]
    for f in files:
        if f.endswith(".npz"):
            continue
        FILES.setdefault(f, os.path.join(root, f))
        if f == "report_labels.csv":
            LABEL_PATHS.append(os.path.join(root, f))
    if "_meta.csv" in files:
        VOL_DIRS["v1"] = root
print(f"Indeks selesai: {len(FILES)} file dalam {time.time()-t0:.0f} detik")
print("Folder volume v1:", VOL_DIRS.get("v1"))

LABEL_PATH = [p for p in LABEL_PATHS if "label-ensemble" in p]
assert LABEL_PATH, f"label-ensemble belum ditambahkan sebagai input. Ditemukan: {LABEL_PATHS}"
LABEL_PATH = LABEL_PATH[0]
print("Label acuan val:", LABEL_PATH)

def find_file(name):
    if name not in FILES:
        raise FileNotFoundError(name)
    return FILES[name]

def mean_auc(y, p):
    """AUC per kelas. Label 0.5 dan kosong diabaikan; label lunak dibinerkan di 0.5."""
    aucs = []
    for k in range(len(LABELS)):
        ok = ~np.isnan(y[:, k]) & (y[:, k] != 0.5)
        yk, pk = (y[ok, k] > 0.5).astype(float), p[ok, k]
        aucs.append(roc_auc_score(yk, pk) if len(np.unique(yk)) == 2 else np.nan)
    return float(np.nanmean(aucs))

Indeks selesai: 134 file dalam 5 detik
Folder volume v1: /kaggle/input/notebooks/kellenvalerie/rsna-knee-02-preprocess-mri/vol_train
Label acuan val: /kaggle/input/notebooks/kellenvalerie/rsna-knee-01d-label-ensemble/report_labels.csv


In [2]:
out_val, out_gold = f"{WORK}/val_preds_effb0.csv", f"{WORK}/gold_preds_effb0.csv"

if os.path.exists(out_val):
    print("Sudah ada, dilewati:", out_val)
else:
    spec = importlib.util.spec_from_file_location("tb", find_file("09_train_student.py"))
    tb = importlib.util.module_from_spec(spec); spec.loader.exec_module(tb)
    torch.set_num_threads(os.cpu_count())

    val_ids = pd.read_csv(find_file("val_preds_effb0_s2.csv"))["StudyInstanceUID"].tolist()
    print("Studi val:", len(val_ids), "| CPU thread:", torch.get_num_threads())

    net = tb.KneeNet("efficientnet_b0", n_views=3, pretrained=False)
    net.load_state_dict(torch.load(find_file("efficientnet_b0_best.pt"), map_location="cpu"))
    net = net.eval().to(memory_format=torch.channels_last)

    ds = tb.KneeDS(val_ids, np.full((len(val_ids), len(LABELS)), np.nan, np.float32),
                   VOL_DIRS["v1"], train=False)
    dl = torch.utils.data.DataLoader(ds, batch_size=4, shuffle=False, num_workers=0)

    ps = []
    with torch.inference_mode():
        for vol, vmask, _, _ in tqdm(dl, desc="Prediksi val effb0"):
            ps.append(torch.sigmoid(net(vol, vmask)).numpy())

    pv = pd.DataFrame(np.concatenate(ps), columns=LABELS)
    pv.insert(0, "StudyInstanceUID", val_ids)
    pv.to_csv(out_val, index=False)
    pd.read_csv(find_file("gold_preds.csv")).to_csv(out_gold, index=False)
    print("Selesai:", out_val, "&", out_gold)

Studi val: 435 | CPU thread: 4


Prediksi val effb0:   0%|          | 0/109 [00:00<?, ?it/s]

Selesai: /kaggle/working/val_preds_effb0.csv & /kaggle/working/gold_preds_effb0.csv


In [3]:
val, gold = {}, {}
for f, path in FILES.items():
    if "_kf" in f:
        continue
    if f.startswith("val_preds_") and f.endswith(".csv"):
        val[f[len("val_preds_"):-4]] = pd.read_csv(path)
    if f.startswith("gold_preds_") and f.endswith(".csv"):
        gold[f[len("gold_preds_"):-4]] = pd.read_csv(path)

val["effb0"] = pd.read_csv(f"{WORK}/val_preds_effb0.csv")
gold["effb0"] = pd.read_csv(f"{WORK}/gold_preds_effb0.csv")

tags = sorted(set(val) & set(gold))
print("Model ditemukan:", tags)

rep = pd.read_csv(LABEL_PATH)
tr = pd.read_csv(f"{COMP}/train.csv")
gold_df = tr[tr[LABELS].notna().all(axis=1)][["StudyInstanceUID"] + LABELS]

val_ids = val[tags[0]]["StudyInstanceUID"]
gold_ids = gold[tags[0]]["StudyInstanceUID"]
for t in tags:
    assert set(val[t]["StudyInstanceUID"]) == set(val_ids), f"split val {t} berbeda"
    assert set(gold[t]["StudyInstanceUID"]) == set(gold_ids), f"studi gold {t} berbeda"

y_val = rep.set_index("StudyInstanceUID").loc[val_ids, LABELS].values.astype(float)
y_gold = gold_df.set_index("StudyInstanceUID").loc[gold_ids, LABELS].values.astype(float)
P_val = {t: val[t].set_index("StudyInstanceUID").loc[val_ids, LABELS].values for t in tags}
P_gold = {t: gold[t].set_index("StudyInstanceUID").loc[gold_ids, LABELS].values for t in tags}
print(f"Studi val: {len(val_ids)} | studi gold: {len(gold_ids)}")

Model ditemukan: ['dinov2s', 'dinov2s_student_d32', 'dinov2s_student_v2', 'dinov2s_student_v3', 'dinov2s_v2', 'effb0', 'effb0_s2', 'effb0_student_d32', 'effb0_student_d32_s7', 'effb0_student_v1', 'effb0_student_v2', 'effb0_student_v2_s7', 'effb0_student_v3', 'effb0_student_v3_s7', 'resnet50']
Studi val: 435 | studi gold: 58


In [4]:
print("Label acuan val:", LABEL_PATH)
print("\n=== Model tunggal (urut gold) ===")
single = pd.DataFrame([{"model": t, "val": mean_auc(y_val, P_val[t]),
                        "gold": mean_auc(y_gold, P_gold[t])} for t in tags])
print(single.sort_values("gold", ascending=False).round(4).to_string(index=False))

POOL = ["effb0", "dinov2s_v2", "effb0_student_v2", "effb0_student_v2_s7", "dinov2s_student_v2",
        "dinov2s_student_v3", "effb0_student_v3", "effb0_student_v3_s7",
        "dinov2s_student_d32", "effb0_student_d32_s7"]
POOL = [t for t in POOL if t in tags]
print("\nPool:", POOL)

def ens(combo, P): return np.mean([P[t] for t in combo], axis=0)

rows = []
for r in range(2, len(POOL) + 1):
    for combo in itertools.combinations(POOL, r):
        rows.append({"combo": " + ".join(combo), "n": r,
                     "val": mean_auc(y_val, ens(combo, P_val)), "gold": mean_auc(y_gold, ens(combo, P_gold))})
res = pd.DataFrame(rows); res["rata2"] = (res["val"] + res["gold"]) / 2
print("\n=== 10 kombinasi terbaik (rata-rata val dan gold) ===")
print(res.sort_values("rata2", ascending=False).head(10).round(4).to_string(index=False))

A = ["effb0", "dinov2s_v2", "effb0_student_v2", "effb0_student_v2_s7", "dinov2s_student_v2"]
B = ["dinov2s_student_v3", "dinov2s_student_v2", "effb0_student_v3", "effb0_student_v3_s7"]
cands = {
    "Ensemble A (LB 0,877)": A,
    "Ensemble B (LB 0,877)": B,
    "B + DINOv2 d32": B + ["dinov2s_student_d32"],
    "B + DINOv2 d32 + EffB0 d32 s7": B + ["dinov2s_student_d32", "effb0_student_d32_s7"],
    "A + B + DINOv2 d32": sorted(set(A + B + ["dinov2s_student_d32"])),
}

def boot_diff(ca, cb, n=1000, seed=0):
    """Selisih gold AUC (cb - ca) dengan bootstrap 58 studi gold."""
    rng = np.random.default_rng(seed)
    pa, pb = ens(ca, P_gold), ens(cb, P_gold)
    d = []
    for _ in range(n):
        i = rng.integers(0, len(y_gold), len(y_gold))
        d.append(mean_auc(y_gold[i], pb[i]) - mean_auc(y_gold[i], pa[i]))
    return np.nanmean(d), np.nanpercentile(d, 2.5), np.nanpercentile(d, 97.5)

print("\n=== Kandidat (selisih gold dibanding Ensemble A, bootstrap 1.000x) ===")
for name, combo in cands.items():
    combo = [t for t in combo if t in tags]
    v, g = mean_auc(y_val, ens(combo, P_val)), mean_auc(y_gold, ens(combo, P_gold))
    if name.startswith("Ensemble A"):
        print(f"{name:32s} val {v:.4f} | gold {g:.4f} | (acuan)")
    else:
        m, lo, hi = boot_diff(A, combo)
        print(f"{name:32s} val {v:.4f} | gold {g:.4f} | selisih gold {m:+.4f} (95% CI {lo:+.4f} .. {hi:+.4f})")

Label acuan val: /kaggle/input/notebooks/kellenvalerie/rsna-knee-01d-label-ensemble/report_labels.csv

=== Model tunggal (urut gold) ===
               model    val   gold
  dinov2s_student_v3 0.8721 0.8538
 dinov2s_student_d32 0.8770 0.8493
    effb0_student_v3 0.8683 0.8484
effb0_student_d32_s7 0.8414 0.8479
  dinov2s_student_v2 0.8468 0.8436
 effb0_student_v3_s7 0.8677 0.8400
               effb0 0.8289 0.8383
 effb0_student_v2_s7 0.8609 0.8361
    effb0_student_v2 0.8671 0.8351
          dinov2s_v2 0.8380 0.8253
            effb0_s2 0.8349 0.8240
    effb0_student_v1 0.8581 0.8182
   effb0_student_d32 0.8268 0.8148
            resnet50 0.8059 0.8036
             dinov2s 0.7086 0.7006

Pool: ['effb0', 'dinov2s_v2', 'effb0_student_v2', 'effb0_student_v2_s7', 'dinov2s_student_v2', 'dinov2s_student_v3', 'effb0_student_v3', 'effb0_student_v3_s7', 'dinov2s_student_d32', 'effb0_student_d32_s7']

=== 10 kombinasi terbaik (rata-rata val dan gold) ===
                                        